# [4] Naive Trial with RoBERTa Compression

## Imports

In [ ]:
import env

In [ ]:
from epidec.datasets import SWUnivDaconDataset

from transformers import pipeline, AutoTokenizer, RobertaForSequenceClassification
from torch.utils.data import DataLoader
from torch.nn import functional as F
from torch import nn
import torch

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import json
import sys

In [ ]:
# Set CUDA Device
device_num = 6

if torch.cuda.is_available() and device_num != -1:
    torch.cuda.set_device(device_num)
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    device_num = -1  # cpu
print(f"INFO: Using device - {device}:{device_num}")

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = SWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = SWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = SWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Model

In [ ]:
class ExtendedRoBERTaDetector(nn.Module):
    model_id = "roberta-large-openai-detector"

    def __init__(self):
        super().__init__()

        # 1. 기존 RoBERTa 모델 (수정 없이 그대로 사용)
        self.roberta_detector = RobertaForSequenceClassification.from_pretrained(
            self.model_id
        )
        for param in self.roberta_detector.parameters():
            param.requires_grad = False

        # RoBERTa의 분류 헤드는 사용하지 않고, 히든 상태만 추출
        # 따라서 RoBERTa를 feature extractor로만 사용
        self.roberta_base = self.roberta_detector.roberta  # 분류 헤드 제거

        hidden_size = self.roberta_base.config.hidden_size  # 768

        # 2. 새로 추가할 Attention Pooling + 분류 레이어
        self.attention_pooling = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1)
        )

        self.final_classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 2)  # AI vs Human
        )

    def forward(self, chunk_input_ids_list, chunk_attention_mask_list):
        """
        chunk_input_ids_list: List[Tensor] - 각 청크의 input_ids
        chunk_attention_mask_list: List[Tensor] - 각 청크의 attention_mask
        """
        chunk_embeddings = []

        # 각 청크를 RoBERTa로 처리
        for input_ids, attention_mask in zip(chunk_input_ids_list, chunk_attention_mask_list):
            with torch.no_grad():  # RoBERTa는 freeze
                outputs = self.roberta_base(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )
                # [CLS] 토큰의 representation 사용
                cls_embedding = outputs.last_hidden_state[0, 0, :]  # [hidden_size]
                chunk_embeddings.append(cls_embedding)

        # 청크 임베딩들을 스택
        chunk_embeddings = torch.stack(chunk_embeddings)  # [num_chunks, hidden_size]

        # Attention Pooling으로 하나의 벡터로 결합
        attention_scores = self.attention_pooling(chunk_embeddings)  # [num_chunks, 1]
        attention_weights = F.softmax(attention_scores, dim=0)

        # 가중평균
        pooled_embedding = (chunk_embeddings * attention_weights).sum(dim=0)  # [hidden_size]

        # 최종 분류
        logits = self.final_classifier(pooled_embedding.unsqueeze(0))  # [1, 2]

        return logits

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ExtendedRoBERTaDetector.model_id)
model = ExtendedRoBERTaDetector()
model.to(device)

In [ ]:
def chunk_text(text, tokenizer=tokenizer, max_length=400):
    words = text.split()
    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:
        word_tokens = len(tokenizer.encode(word, add_special_tokens=False))
        if current_length + word_tokens > max_length and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = [word]
            current_length = word_tokens
        else:
            current_chunk.append(word)
            current_length += word_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

## Train and Evaluation

In [ ]:
learning_rate = 1e-6

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
check_only_for = None
threshold = 0.5

In [ ]:
# Training
fine_tune_amount = 200

count_human, count_ai = 0, 0
corrects, errors, true_human, false_human, results = [], [], [], [], []
progress = tqdm(DataLoader(train_dataset, batch_size=1, shuffle=True), desc="Training...")

for idx, data in enumerate(progress):
    label = data[1][0]
    if label == 0:
        if count_human >= fine_tune_amount:
            continue  # Skip if we have already fine-tuned enough human data
        count_human += 1
    else:
        if count_ai >= fine_tune_amount:
            continue  # Skip if we have already fine-tuned enough AI data
        count_ai += 1
    data = data[0][0]
    optimizer.zero_grad()

    if check_only_for is not None and label != check_only_for:
        continue  # Skip if the label does not match the specified check

    chunks = chunk_text(data)
    chunk_inputs = []
    chunk_masks = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            max_length=512,
            truncation=True,
            padding=True
        )
        chunk_inputs.append(inputs['input_ids'].to(device))
        chunk_masks.append(inputs['attention_mask'].to(device))

    logits = model(chunk_inputs, chunk_masks)
    loss = criterion(logits, torch.tensor([[label, 1-label]]).float().to(device))
    loss.backward()
    optimizer.step()

    probs = torch.softmax(logits, dim=-1)
    predicted = probs[0][1].item()

    result = dict(question=data, label=label, predicted=predicted)
    predicted_label = 1 if predicted >= threshold else 0
    if predicted_label == label:
        corrects.append(result)
        if label == 0:
            true_human.append(result)
    else:
        errors.append(result)
        if label == 0:
            false_human.append(result)
    results.append(result)
    progress.set_description(f"Correct: {len(corrects)/len(results):.6%} [H: {len(true_human)}, A: {len(corrects)-len(true_human)}], Errors: {len(errors)} [H: {len(false_human)}, A: {len(errors)-len(false_human)}]")

print(f"INFO: Correct: {len(corrects)}/{len(results)}, Errors: {len(errors)}/{len(results)}")

In [ ]:
# Validation
corrects, errors, true_human, false_human, results = [], [], [], [], []
progress = tqdm(DataLoader(valid_dataset, batch_size=1, shuffle=True), desc="Validating...")

for idx, data in enumerate(progress):
    label = data[1][0]
    data = data[0][0]

    if check_only_for is not None and label != check_only_for:
        continue  # Skip if the label does not match the specified check

    chunks = chunk_text(data)
    chunk_inputs = []
    chunk_masks = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            max_length=512,
            truncation=True,
            padding=True
        )
        chunk_inputs.append(inputs['input_ids'].to(device))
        chunk_masks.append(inputs['attention_mask'].to(device))

    with torch.no_grad():
        logits = model(chunk_inputs, chunk_masks)
        probs = torch.softmax(logits, dim=-1)
        predicted = probs[0][1].item()

    result = dict(question=data, label=label, predicted=predicted)
    predicted_label = 1 if predicted >= threshold else 0
    if predicted_label == label:
        corrects.append(result)
        if label == 0:
            true_human.append(result)
    else:
        errors.append(result)
        print(f"ERROR: Incorrect prediction for index {idx}, expected {label}, got {predicted}")
        if label == 0:
            false_human.append(result)
    results.append(result)
    progress.set_description(f"Correct: {len(corrects)/len(results):.6%} [H: {len(true_human)}, A: {len(corrects)-len(true_human)}], Errors: {len(errors)} [H: {len(false_human)}, A: {len(errors)-len(false_human)}]")

print(f"INFO: Correct: {len(corrects)}/{len(results)}, Errors: {len(errors)}/{len(results)}")

In [ ]:
# Test
results = []
for idx, data in enumerate(tqdm(test_dataset, desc="Testing...")):
    data = data[0]
    chunks = chunk_text(data)
    chunk_inputs = []
    chunk_masks = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            max_length=512,
            truncation=True,
            padding=True
        )
        chunk_inputs.append(inputs['input_ids'])
        chunk_masks.append(inputs['attention_mask'])

    with torch.no_grad():
        logits = model(chunk_inputs, chunk_masks)
        probs = torch.softmax(logits, dim=-1)
        predicted = probs[0][1].item()

    result = dict(question=data, label=predicted)
    results.append(result)

In [ ]:
r = [1 if results['label'] >= threshold else 0 for results in results]

In [ ]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')

In [ ]:
sub

In [ ]:
sub['generated'] = r

In [ ]:
sub

In [ ]:
sub.to_csv("./data/submission_roberta.csv", index=False, encoding='utf-8-sig')